### How Spark Performs Joins

node-to-node communication strategy 
per node computation strategy.

### Big table–to–big table

When you join a big table to another big table, you end up with a shuffle join,

In a shuffle join, every node talks to every other node and they share data according to which
node has a certain key or set of keys (on which you are joining). These joins are expensive
because the network can become congested with traffic, especially if your data is not partitioned
well.

## Big table–to–small table

When the table is small enough to fit into the memory of a single worker node, with some
breathing room of course, we can optimize our join. 
often be more efficient to use a `broadcast join`


What this
means is that we will replicate our small DataFrame onto every worker node in the cluster (be it
located on one machine or many). Now this sounds expensive. However, what this does is
prevent us from performing the all-to-all communication during the entire join process. Instead,
we perform it only once at the beginning and then let each individual worker node perform the
work without having to wait or communicate with any other worker node,

In [1]:
%run ../setup_data.ipynb

No active 'spark' session found to stop.
Starting new Spark session: 'working_with_different_data_type'...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/12 13:49:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


flight path verified (6 items): /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/flight-data/csv
retail path verified (7 items): /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/retail-data
Loading flight data...
Loading retail data...
+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|sujeet_db|   department|      false|
|sujeet_db|       flight|      false|
|sujeet_db|      payment|      false|
|sujeet_db|       retail|      false|
|sujeet_db|sales_metrics|      false|
|sujeet_db| skew_metrics|      false|
|sujeet_db|weather_sales|      false|
+---------+-------------+-----------+

@@@@@@@@@@@@@@@@@
creating dataset 'emp_df' and 'dept_df' for join playbook
@@@@@@@@@@@@@@@@@@
root
 |-- emp_id: integer (nullable = true)
 |-- emp_name: string (nullable = true)
 |-- dept_id: integer (nullable = true)
 |-- details: struct (nullable = true)
 |    |-- location: string (nullable = true)
 |    

In [ ]:
from pyspark.sql.functions import broadcast

## BroadcastHashJoin
emp_df.join(broadcast(dept_df), emp_df['dept_id'] == dept_df['dept_id']).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [dept_id#114], [dept_id#117], Inner, BuildRight, false, false
   :- Filter isnotnull(dept_id#114)
   :  +- Scan ExistingRDD[emp_id#112,emp_name#113,dept_id#114,details#115,skills#116]
   +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=202]
      +- Filter isnotnull(dept_id#117)
         +- Scan ExistingRDD[dept_id#117,dept_name#118,hq_info#119,req_skills#120]




In [ ]:
## SortMergeJoin
emp_df.join(dept_df, emp_df['dept_id'] == dept_df['dept_id']).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [dept_id#114], [dept_id#117], Inner
   :- Sort [dept_id#114 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(dept_id#114, 200), ENSURE_REQUIREMENTS, [plan_id=225]
   :     +- Filter isnotnull(dept_id#114)
   :        +- Scan ExistingRDD[emp_id#112,emp_name#113,dept_id#114,details#115,skills#116]
   +- Sort [dept_id#117 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(dept_id#117, 200), ENSURE_REQUIREMENTS, [plan_id=226]
         +- Filter isnotnull(dept_id#117)
            +- Scan ExistingRDD[dept_id#117,dept_name#118,hq_info#119,req_skills#120]




## Little table–to–little table
When performing joins with small tables, it’s usually best to let Spark decide how to join them